# Praktikum Computer Vision: Stereo Vision dengan **1 Smartphone Digeser**

**Tujuan:** Menggunakan satu HP sebagai dua posisi kamera virtual (**kiri** dan **kanan**), melakukan **kalibrasi untuk memperoleh focal length dalam piksel**, mengukur *disparity*, dan memperkirakan jarak benda.

**Alur pembelajaran:** Persiapan → foto *chessboard* → kalibrasi kamera \(f_x,f_y\) → dua foto stereo dengan HP digeser → cek keselarasan → disparity → estimasi jarak → evaluasi → **tugas flowchart**.

> **Penting:** Ini **stereo sequential**, bukan dua foto serentak. Hanya untuk **objek diam**, HP tidak boleh diputar saat digeser, pengaturan kamera harus tetap (lensa, fokus, zoom, resolusi). Jarak geser horizontal **B** diukur antara dua posisi pusat kamera. Nilai kedalaman akurat hanya jika asumsi geometri dan kalibrasi cukup baik.

**Platform:** Google Colab atau Jupyter Notebook lokal. Notebook ini tidak membutuhkan akses kamera langsung: unggah hasil foto dari HP.

## 1 — Gambaran alat dan langkah foto

- **HP 1 buah**, penyangga/rel geser lurus (bisa dibuat dari penggaris panjang + dudukan), **penggaris atau meteran**, pola kotak catur yang dicetak datar dan **objek diam bertekstur** (misalnya kotak dengan stiker).
- **Kalibrasi**: gunakan papan catur dengan **9 × 6 sudut dalam** (berarti **10 × 7 kotak**), setiap kotak misalnya **25 mm**; **ukur kotak hasil cetak** memakai penggaris. Ambil **15–25 foto papan** dari sudut/posisi berbeda dan berbagai bagian frame. Pada tahap ini HP boleh diubah arah untuk mendapatkan variasi posisi papan. Semua foto harus dari **kamera/lensa/pengaturan yang sama**.
- **Foto stereo**: letakkan HP di posisi **L** → foto `left.jpg` → geser **ke kanan 10 cm** tanpa memutar/menaikkan HP → foto `right.jpg`. Objek dan latar tetap diam.
- Lakukan foto stereo setelah selesai kalibrasi tanpa mengganti konfigurasi kamera. Jangan gunakan mode portrait, digital zoom, atau perpindahan otomatis antar-lensa.

**Struktur data di folder atau ZIP:**
```text
praktikum_stereo/
├── calibration/
│   ├── cal_01.jpg
│   ├── cal_02.jpg
│   └── ... (15–25 foto)
└── stereo/
    ├── left.jpg
    └── right.jpg
```

**Bedakan dua tahap:** foto `calibration/` untuk mencari **f** dan distorsi; `left.jpg/right.jpg` untuk mencari **d**. Jarak geser **B tidak ditemukan dari kalibrasi satu kamera**: harus diukur dengan penggaris.

In [ ]:
# 1A — Ilustrasi prosedur (dihasilkan langsung oleh kode)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
fig, ax = plt.subplots(figsize=(12, 4.4))
ax.set_xlim(0, 12); ax.set_ylim(0, 4.4); ax.axis('off')
ax.plot([0.5, 7.2], [1,1], lw=9, color='gray', alpha=.45)
for x, name, color in [(1.8,'L: foto pertama','royalblue'),(5.4,'R: foto kedua','firebrick')]:
    ax.add_patch(Rectangle((x-.38,1.1),.76,1.35,fill=False,edgecolor=color,lw=3))
    ax.add_patch(Circle((x,2.12),.14,color=color))
    ax.text(x,2.75,name,ha='center',color=color,fontsize=11,weight='bold')
ax.annotate('',xy=(5.4,.55),xytext=(1.8,.55),arrowprops={'arrowstyle':'<->','lw':2})
ax.text(3.6,.2,'B = jarak geser, contoh 10 cm',ha='center',fontsize=11)
ax.add_patch(Rectangle((9.0,1.1),1.35,1.7,fill=False,lw=2,color='seagreen'))
for i in range(1,5):
    ax.plot([9+1.35*i/5]*2,[1.1,2.8],color='seagreen',alpha=.45)
ax.text(9.65,3.15,'OBJEK DIAM',ha='center',fontsize=12,weight='bold',color='seagreen')
ax.plot([1.8,9.3],[2.1,2.1],ls=':',color='royalblue');ax.plot([5.4,9.3],[2.1,2.1],ls=':',color='firebrick')
ax.text(6.4,4.0,'HP digeser lurus; orientasi tetap, objek tidak bergerak',ha='center',fontsize=13)
plt.show()

## 2 — Mengapa kalibrasi bisa menghasilkan \(f_x\) (misalnya 800 px)?

Pada model kamera pinhole, koordinat suatu titik 3D ke citra mengikuti:

\[
 u=f_xrac{X}{Z}+c_x,\qquad v=f_yrac{Y}{Z}+c_y
\]

- **\(X,Y,Z\)**: posisi titik di depan kamera dalam satuan fisik.
- **\(u,v\)**: posisi titik pada foto dalam **piksel**.
- **\(f_x,f_y\)**: panjang fokus efektif dalam **piksel** — **ini yang kita cari dari kalibrasi**.
- **\(c_x,c_y\)**: posisi *principal point*, biasanya dekat pusat gambar, **bukan** panjang fokus.

Pada papan catur, jarak antar-sudut **diketahui** (contoh 25 mm). OpenCV mendeteksi posisi sudut pada foto (**piksel**) lalu mencari parameter \(f_x,f_y,c_x,c_y\), orientasi papan pada tiap foto, dan distorsi yang **meminimalkan selisih proyeksi**. Itulah asal angka seperti 800 px: **hasil optimasi berdasarkan banyak pasangan titik 3D–2D**, bukan angka yang ditetapkan terlebih dahulu. Angka nyatanya bergantung kamera dan resolusi.

**Contoh intuisi sederhana** (bukan pengganti kalibrasi penuh): jika suatu perpindahan 100 mm sejajar sensor pada kedalaman 1000 mm tampak 80 px pada gambar, \(fpprox 80	imes1000/100=800\) px.

**Catatan:** satuan kotak chessboard dalam mm terutama mempengaruhi hasil pose/translasi dalam mm. Estimasi \(f\) dalam px diperoleh dari geometri proyeksi di berbagai tampilan. Ukuran kotak tetap harus dicatat secara konsisten.

In [ ]:
# 2A — Diagram asal f: kesebangunan segitiga (skema bukan skala fisik)
fig,ax=plt.subplots(figsize=(11,4)); ax.set_xlim(-1.2,10);ax.set_ylim(-3.3,3.3);ax.axis('off')
O=(2.3,0);sensor_x=3.8;plane_x=8.7
ax.axhline(0,ls='--',lw=1,color='gray')
ax.plot([sensor_x,sensor_x],[-2.5,2.5],color='slategray',lw=4)
ax.plot([plane_x,plane_x],[-2.7,2.7],color='seagreen',lw=4)
ax.plot([O[0],plane_x],[0,2],color='royalblue',lw=2)
ax.plot([O[0],sensor_x],[0,-2*(sensor_x-O[0])/(plane_x-O[0])],color='firebrick',lw=2)
ax.scatter([O[0],plane_x,sensor_x],[0,2,-2*(sensor_x-O[0])/(plane_x-O[0])],s=40)
ax.annotate('',(sensor_x,-2.95),(O[0],-2.95),arrowprops={'arrowstyle':'<->'})
ax.text(3.05,-3.25,'f (ke bidang citra)',ha='center')
ax.annotate('',(plane_x,2.65),(O[0],2.65),arrowprops={'arrowstyle':'<->'})
ax.text(5.5,2.85,'Z (jarak objek)',ha='center')
ax.text(1.9,.35,'O: pusat proyeksi')
ax.text(sensor_x+.15,-1,'x: offset proyeksi',color='firebrick')
ax.text(plane_x+.12,1,'X: offset objek',color='seagreen')
ax.text(4.8,-2.15,'f/Z = x/X  (besar nilai)\nContoh: f = 80 px x 1000 mm / 100 mm = 800 px',ha='center',fontsize=11)
plt.show()

## 3 — Persiapan Google Colab / Jupyter

1. **Google Colab:** jalankan sel `uploads ZIP` dan unggah file ZIP berstruktur seperti di atas, atau salin folder ke `/content/praktikum_stereo`.
2. **Jupyter lokal:** letakkan folder `praktikum_stereo` di folder kerja notebook, lalu set `DATA_DIR = Path('praktikum_stereo')`.
3. Ukuran foto kalibrasi **harus konsisten** dan sama dengan foto stereo. Bila foto berasal dari rotasi otomatis EXIF atau resolusi yang berubah, rapikan dulu secara konsisten.
4. Pada pengambilan foto, disarankan kunci fokus dan exposure agar parameter lensa tidak berubah.

**Tidak ada hasil depth yang otomatis benar tanpa foto asli.** Semua nilai akhir dihitung dari file Anda sendiri.

In [ ]:
# 3A — Dependencies + konfigurasi
# Di Colab, jika cv2 belum ada, jalankan: %pip install opencv-python-headless matplotlib numpy
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt, json, zipfile, os
print('OpenCV:', cv2.__version__)

# UBAH SESUAI HASIL CETAK / FOTO
DATA_DIR = Path('/content/praktikum_stereo') if Path('/content').exists() else Path('praktikum_stereo')
CALIB_DIR = DATA_DIR / 'calibration'
STEREO_DIR = DATA_DIR / 'stereo'
OUTPUT_DIR = DATA_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATTERN_SIZE = (9, 6)   # jumlah SUDUT DALAM: kolom, baris (bukan kotak)
SQUARE_SIZE_MM = 25.0  # ukur dengan penggaris setelah dicetak!
BASELINE_CM = 10.0     # jarak pusat kamera posisi L ke R
MIN_CALIB_IMAGES = 10  # lebih baik 15–25 foto berkualitas
print('Data folder:', DATA_DIR.resolve())

In [ ]:
# 3B — OPSIONAL KHUSUS COLAB: unggah ZIP dari HP/laptop
# Struktur ZIP: calibration/*.jpg dan stereo/left.jpg + stereo/right.jpg
# Jalankan hanya bila ingin upload ZIP lewat browser Colab.
UPLOAD_ZIP_DI_COLAB = False
if UPLOAD_ZIP_DI_COLAB:
    try:
        from google.colab import files
        uploaded = files.upload()
        for filename, contents in uploaded.items():
            if filename.lower().endswith('.zip'):
                import io
                with zipfile.ZipFile(io.BytesIO(contents)) as z:
                    for info in z.infolist():
                        p = Path(info.filename)
                        if p.is_absolute() or '..' in p.parts:
                            raise ValueError('ZIP tidak aman: ' + info.filename)
                    z.extractall(DATA_DIR)
                print('Diekstrak ke:', DATA_DIR)
    except ImportError:
        print('Upload browser khusus Google Colab; gunakan folder lokal pada Jupyter.')

## 4 — Periksa foto kalibrasi

OpenCV mencari **9 × 6 titik persilangan internal** pada setiap foto. Jika pola 9 × 6 gagal, periksa apakah papan yang dicetak benar-benar **10 × 7 kotak**. Foto diambil dari berbagai kemiringan dan posisi, tetapi seluruh papan harus terlihat, tajam, dan tidak melengkung.

In [ ]:
# 4A — Temukan sudut chessboard
extensions = {'.jpg','.jpeg','.png','.bmp'}
cal_paths = sorted(p for p in CALIB_DIR.glob('*') if p.suffix.lower() in extensions)
if not cal_paths:
    print('BELUM ADA FOTO. Taruh 15–25 foto di', CALIB_DIR)
    print('Contoh:', CALIB_DIR/'cal_01.jpg')
else:
    obj_template = np.zeros((PATTERN_SIZE[0]*PATTERN_SIZE[1],3),dtype=np.float32)
    obj_template[:,:2] = np.mgrid[0:PATTERN_SIZE[0],0:PATTERN_SIZE[1]].T.reshape(-1,2)
    obj_template *= SQUARE_SIZE_MM
    obj_points, img_points, good_paths = [],[],[]
    image_wh = None
    fig,axes = plt.subplots(2,4,figsize=(13,6)); axes=axes.ravel()
    for i,path in enumerate(cal_paths):
        img=cv2.imread(str(path));
        if img is None: print('Gagal dibaca:',path.name); continue
        gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        wh=(gray.shape[1],gray.shape[0])
        if image_wh is None: image_wh=wh
        if wh != image_wh:
            print('Lewati ukuran berbeda:',path.name,wh); continue
        found,corners=cv2.findChessboardCorners(gray,PATTERN_SIZE,cv2.CALIB_CB_ADAPTIVE_THRESH+cv2.CALIB_CB_NORMALIZE_IMAGE)
        if found:
            corners=cv2.cornerSubPix(gray,corners,(11,11),(-1,-1),
                     (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,30,.001))
            obj_points.append(obj_template.copy());img_points.append(corners);good_paths.append(path)
        if i<8:
            vis=img.copy()
            if found: cv2.drawChessboardCorners(vis,PATTERN_SIZE,corners,True)
            axes[i].imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB)); axes[i].set_title(f'{path.name}: {"OK" if found else "GAGAL"}',fontsize=9);axes[i].axis('off')
    for a in axes[min(len(cal_paths),8):]:a.axis('off')
    plt.tight_layout();plt.show()
    print(f'Sudut terdeteksi: {len(good_paths)}/{len(cal_paths)} foto. Ukuran: {image_wh}')
    if len(good_paths)<MIN_CALIB_IMAGES:print('⚠️ Foto valid masih sedikit; ambil ulang sebagian foto.')

In [ ]:
# 4B — Estimasi K, distorsi dan f dalam piksel
if not cal_paths or 'good_paths' not in globals() or len(good_paths)<MIN_CALIB_IMAGES:
    print('BELUM BISA KALIBRASI: kumpulkan minimal',MIN_CALIB_IMAGES,'foto chessboard terdeteksi.')
else:
    rms,K,dist,rvecs,tvecs=cv2.calibrateCamera(obj_points,img_points,image_wh,None,None)
    fx,fy,cx,cy = float(K[0,0]),float(K[1,1]),float(K[0,2]),float(K[1,2])
    print('K (camera intrinsic matrix):\n',K)
    print(f'fx = {fx:.2f} px, fy = {fy:.2f} px')
    print(f'cx = {cx:.2f} px, cy = {cy:.2f} px (titik pusat proyeksi, bukan f)')
    print(f'RMS reprojection error = {rms:.3f} px (lebih rendah biasanya lebih baik)')
    print('Koefisien distorsi:',dist.ravel())
    np.savez(OUTPUT_DIR/'calibration.npz',K=K,dist=dist,image_wh=image_wh,
             rms=rms,square_mm=SQUARE_SIZE_MM)
    errors=[]
    for o,im,r,t in zip(obj_points,img_points,rvecs,tvecs):
        projected,_=cv2.projectPoints(o,r,t,K,dist)
        errors.append(float(np.mean(np.linalg.norm(im.reshape(-1,2)-projected.reshape(-1,2),axis=1))))
    print(f'Mean corner reprojection error = {np.mean(errors):.3f} px')
    fig,ax=plt.subplots(figsize=(9,3));ax.bar(range(1,len(errors)+1),errors)
    ax.set(xlabel='Foto kalibrasi valid ke-',ylabel='Mean error (px)',title='Diagnostik: reprojection error per foto');plt.show()

## 5 — Foto kiri/kanan dan pemeriksaan keselarasan

**Syarat:** `left.jpg` = posisi HP sebelum digeser; `right.jpg` = HP setelah digeser **ke kanan**. Baseline diketahui dari penggaris, misalnya **10 cm**. Papan kalibrasi boleh diambil sebelum ini; untuk foto stereo gunakan **objek diam**.

Pada dua kamera virtual yang **ideal paralel**, titik objek yang sama memiliki koordinat vertikal hampir sama: \(v_Lpprox v_R\). Jika banyak titik tidak sebidang horizontal, pengambilan stereo harus diulang dengan rel lebih stabil. **Kalibrasi satu kamera tidak dapat secara otomatis mengetahui rotasi/arah geser nyata.** Kita memakai asumsi geser horizontal murni, lalu memeriksanya.

In [ ]:
# 5A — Baca pasangan foto, koreksi distorsi, cek resolusi
lp,rp=STEREO_DIR/'left.jpg',STEREO_DIR/'right.jpg'
if not (lp.exists() and rp.exists()):
    print('BELUM ADA FOTO STEREO. Simpan ke:', lp, 'dan',rp)
elif not (OUTPUT_DIR/'calibration.npz').exists():
    print('Jalankan kalibrasi (sel 4B) lebih dulu.')
else:
    c=np.load(OUTPUT_DIR/'calibration.npz');K=c['K'];dist=c['dist'];image_wh=tuple(c['image_wh'].tolist())
    left=cv2.imread(str(lp));right=cv2.imread(str(rp))
    assert left is not None and right is not None,'JPG gagal dibaca'
    assert (left.shape[1],left.shape[0])==image_wh and left.shape==right.shape,        'Resolusi foto stereo wajib sama dengan foto kalibrasi, dan kiri=kanan'
    # Koreksi distorsi memakai K asli: koordinat hasil tetap dalam pixel grid yang sama.
    left_u=cv2.undistort(left,K,dist,None,K);right_u=cv2.undistort(right,K,dist,None,K)
    fig,ax=plt.subplots(1,2,figsize=(13,5))
    for a,img,title in zip(ax,[left_u,right_u],['Left (L) - undistorted','Right (R) - undistorted']):
        a.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB));a.set_title(title);a.axis('off')
    plt.tight_layout();plt.show()

In [ ]:
# 5B — Periksa garis epipolar (selisih y) dengan fitur ORB
if 'left_u' not in globals():
    print('Belum tersedia pasangan stereo yang valid.')
else:
    gL=cv2.cvtColor(left_u,cv2.COLOR_BGR2GRAY)
    gR=cv2.cvtColor(right_u,cv2.COLOR_BGR2GRAY)
    orb=cv2.ORB_create(nfeatures=3000)
    kL,desL=orb.detectAndCompute(gL,None);kR,desR=orb.detectAndCompute(gR,None)
    matches=[]
    if desL is not None and desR is not None:
        pairs=cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(desL,desR,k=2)
        matches=[m for pair in pairs if len(pair)==2 for m,n in [pair] if m.distance<.70*n.distance]
    print('Good ORB matches:',len(matches))
    if len(matches)>=12:
        pL=np.float32([kL[m.queryIdx].pt for m in matches]); pR=np.float32([kR[m.trainIdx].pt for m in matches])
        # Filter kandidat dominan: gerak translasi gambar (untuk visualisasi diagnostik saja)
        A,inliers=cv2.estimateAffinePartial2D(pR,pL,method=cv2.RANSAC,ransacReprojThreshold=3)
        keep=(inliers.ravel()>0) if inliers is not None else np.ones(len(matches),dtype=bool)
        yerr=np.abs(pL[keep,1]-pR[keep,1])
        signed_disp=pL[keep,0]-pR[keep,0]
        print('Median |yL-yR| (px):',round(float(np.median(yerr)),2))
        print('Median (xL-xR) (px):',round(float(np.median(signed_disp)),2))
        print('Matches lolos RANSAC:',int(np.sum(keep)))
        if np.median(yerr)>2: print('⚠️ Vertikal tidak sejajar. Ulang foto; jangan hanya memaksa warping tanpa mengoreksi geometri.')
        if np.median(signed_disp)<=0:print('⚠️ Disparity tidak positif. Periksa left/right dan arah geser HP.')
        select=[matches[i] for i in np.flatnonzero(keep)[:35]]
        preview=cv2.drawMatches(left_u,kL,right_u,kR,select,None,flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        plt.figure(figsize=(14,6));plt.imshow(cv2.cvtColor(preview,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()
    else: print('Tidak cukup fitur: tambah tekstur/cahaya, pastikan scene tumpang tindih.')

## 6 — Metode A (paling transparan): hitung **satu titik** secara manual

Cari **objek/titik yang sama** di gambar kiri dan kanan. Koordinatnya dibaca dalam piksel: \((x_L,y_L)\), \((x_R,y_R)\).

\[
 d=x_L-x_R,\qquad Z=rac{f_x\,B}{d}
\]

Jika \(f_x\) dalam **px**, \(B\) dalam **meter** dan \(d\) dalam **px**, maka \(Z\) dalam **meter**. Rumus ini untuk pasangan **terdistorsi telah dikoreksi, paralel, dan ter-rektifikasi (atau nyaris sejajar)**.

> Contoh untuk penjelasan, **bukan hasil kamera Anda**: \(f_x=800\) px; \(B=0.10\) m; \(x_L=350\); \(x_R=310\) → \(d=40\) px → \(Z=2\) m.

In [ ]:
# 6A — Hitung contoh, kemudian ganti koordinat dengan hasil pengamatan Anda
# Nilai ini CONTOH / DEMO. Ubah empat angka bila titik sudah diketahui.
xL_demo, yL_demo = 350, 240
xR_demo, yR_demo = 310, 240
f_demo, B_demo = 800.0, 0.10

def depth_from_disparity(f_px,baseline_m,x_left,x_right):
    d=float(x_left)-float(x_right)
    if d<=0: raise ValueError('Disparity harus positif. Periksa urutan L/R dan titik yang dipilih.')
    return d,float(f_px)*float(baseline_m)/d

demo_d,demo_z=depth_from_disparity(f_demo,B_demo,xL_demo,xR_demo)
print(f'CONTOH: disparity = {demo_d:.1f} px; depth = {demo_z:.3f} m')
print('Apakah y kiri dan kanan sama?', yL_demo==yR_demo)
if 'K' in globals() and 'left_u' in globals():
    print('Focal length kalibrasi Anda:',round(float(K[0,0]),2),'px')
    print('Untuk menghitung titik nyata: depth_from_disparity(K[0,0], BASELINE_CM/100, xL, xR)')

## 7 — Metode B: disparity map otomatis dengan OpenCV StereoSGBM

SGBM mencari pasangan blok/pola sepanjang garis horizontal. Hasilnya **disparity untuk banyak piksel**; nilai depth diperoleh dari \(Z=f_x B/d\). **Persyaratan:** foto **nyaris rectified** setelah koreksi distorsi. Ketidaksejajaran akan merusak disparity map. Kalau validasi tahap 5B gagal, ulang pengambilan atau lakukan estimasi pose & rectification yang benar.

**Pengaturan awal:** `numDisparities` kelipatan 16, `blockSize` bilangan ganjil; parameter bisa dituning. Hasil `StereoSGBM.compute` dibagi 16 sebelum digunakan dalam rumus jarak.

In [ ]:
# 7A — StereoSGBM + depth map
if 'gL' not in globals():
    print('Belum ada input valid; sel ini akan dijalankan setelah 5A–5B.')
else:
    block_size=7
    num_disp=min(128,max(16,(gL.shape[1]//4//16)*16))
    sgbm=cv2.StereoSGBM_create(minDisparity=0,numDisparities=num_disp,blockSize=block_size,
       P1=8*block_size**2,P2=32*block_size**2,disp12MaxDiff=1,
       uniquenessRatio=10,speckleWindowSize=100,speckleRange=2)
    disparity=sgbm.compute(gL,gR).astype(np.float32)/16.0
    # Batas validitas sesuai konvensi disparity positif dan jangkauan pencarian
    valid=(disparity>1.0) & (disparity<num_disp)
    depth_m=np.full(disparity.shape,np.nan,dtype=np.float32)
    fx_real=float(K[0,0]); B_m=BASELINE_CM/100.0
    depth_m[valid]=fx_real*B_m/disparity[valid]
    print('fx hasil kalibrasi =',round(fx_real,2),'px; B =',B_m,'m')
    print('Persentase disparity valid:',round(100*valid.mean(),1),'%')
    disp_vis=np.ma.masked_where(~valid,disparity)
    z_vis=np.ma.masked_where(~valid,depth_m)
    fig,axs=plt.subplots(1,3,figsize=(17,5))
    axs[0].imshow(cv2.cvtColor(left_u,cv2.COLOR_BGR2RGB));axs[0].set_title('Input kiri')
    a=axs[1].imshow(disp_vis,cmap='turbo',vmin=0,vmax=num_disp);axs[1].set_title('Disparity (px)');fig.colorbar(a,ax=axs[1],shrink=.72)
    a=axs[2].imshow(z_vis,cmap='viridis',vmin=.25,vmax=5);axs[2].set_title('Depth (m; rentang tampilan 0.25–5)');fig.colorbar(a,ax=axs[2],shrink=.72)
    for a in axs:a.axis('off')
    plt.tight_layout();plt.show()
    np.save(OUTPUT_DIR/'disparity_px.npy',disparity)
    np.save(OUTPUT_DIR/'depth_m.npy',depth_m)
    cv2.imwrite(str(OUTPUT_DIR/'left_undistorted.png'),left_u)
    print('Output tersimpan:',OUTPUT_DIR)

## 8 — Mengukur depth pada titik / ROI tertentu

Tentukan \((x,y)\) pada gambar kiri hasil koreksi distorsi. Jangan mengukur pada tekstur datar, permukaan reflektif, tepian tajam atau area tanpa disparity valid. Lebih baik ambil **median ROI** kecil daripada satu piksel. `DEPTH_REFERENSI_M` adalah hasil meteran dari posisi kamera L sepanjang arah pandang, bukan nilai dari kode.

**Ingat:** kamera single-slider mengandalkan asumsi translasi murni. Jika hasil buruk meski kalibrasi baik, periksa rotasi HP dan baseline.

In [ ]:
# 8A — Pilih koordinat titik (ubah setelah melihat gambar pada sel 7A)
ROI_X = 320                 # contoh: ubah sesuai objek pada foto
ROI_Y = 240
ROI_HALF = 7
DEPTH_REFERENSI_M = None     # contoh 1.50 setelah diukur dengan meteran
if 'depth_m' not in globals():
    print('Jalankan penghitungan depth map lebih dulu.')
else:
    H,W=depth_m.shape
    if not(0<=ROI_X<W and 0<=ROI_Y<H):
        print(f'Koordinat di luar citra: resolusi {W} x {H}')
    else:
        x0,x1=max(0,ROI_X-ROI_HALF),min(W,ROI_X+ROI_HALF+1)
        y0,y1=max(0,ROI_Y-ROI_HALF),min(H,ROI_Y+ROI_HALF+1)
        vals=depth_m[y0:y1,x0:x1]
        vals=vals[np.isfinite(vals)]
        if len(vals)==0:print('Tidak ada disparity valid di ROI; pilih objek lain atau perbaiki foto.')
        else:
            pred=float(np.median(vals))
            print(f'Median depth ROI = {pred:.3f} m ({len(vals)} piksel valid)')
            if DEPTH_REFERENSI_M is not None:
                ref=float(DEPTH_REFERENSI_M)
                print('Error absolut (m) =',round(abs(pred-ref),3))
                print('Error relatif (%) =',round(100*abs(pred-ref)/ref,2))
            fig,ax=plt.subplots(figsize=(9,6));ax.imshow(cv2.cvtColor(left_u,cv2.COLOR_BGR2RGB))
            ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,color='yellow',lw=2))
            ax.scatter([ROI_X],[ROI_Y],c='red',s=60);ax.set_title(f'ROI pengukuran: {pred:.2f} m');ax.axis('off');plt.show()

## 9 — Percobaan wajib dan interpretasi

Lakukan minimal **3 jarak objek** (misal 0,5 m; 1,0 m; 1,5 m) pada baseline tetap 10 cm. Untuk setiap jarak, simpan pasangan stereo, jalankan kembali tahap 5–8, dan catat hasil. Agar praktis, ganti `left.jpg`/`right.jpg` untuk tiap percobaan atau buat salinan folder per eksperimen.

| Percobaan | Baseline B (cm) | Jarak meteran Z (m) | Disparity d (px) | Estimasi Z (m) | Error (%) |
|---|---:|---:|---:|---:|---:|
| 1 | 10 | 0,50 | ... | ... | ... |
| 2 | 10 | 1,00 | ... | ... | ... |
| 3 | 10 | 1,50 | ... | ... | ... |

**Jawab dalam laporan:** Apakah objek lebih dekat menghasilkan disparity lebih besar? Mengapa bidang polos menyulitkan matching? Apa dampak baseline yang salah ukur? Bagaimana rotasi HP memengaruhi hasil? Apakah error meningkat untuk objek jauh?

**Catatan evaluasi:** kesalahan kalibrasi, distorsi yang belum terkoreksi, blur, auto focus, perubahan zoom, perbedaan pencahayaan, gambar tak sejajar, serta pengukuran baseline bukan antarpusat kamera bisa menyebabkan error.

In [ ]:
# 9A — Kalkulator cepat perbandingan teori B dan Z
if 'K' in globals():
    fx_use=float(K[0,0]);print(f'Menggunakan fx kalibrasi: {fx_use:.1f} px')
else:
    fx_use=800.0;print('DEMO SAJA: fx=800 px; bukan hasil kalibrasi')
for z in [0.5,1.0,1.5,2.0]:
    print(f'Z nyata {z:.2f} m -> disparity ideal ≈ {fx_use*(BASELINE_CM/100)/z:.1f} px')

## 10 — TUGAS: buat **flowchart dari kode yang benar-benar dijalankan**

**Tugas individu/kelompok:** Gambarkan dua flowchart berdasarkan notebook ini, menggunakan diagrams.net (draw.io), PowerPoint, atau Mermaid. **Jangan hanya menyalin flowchart contoh**: sertakan keputusan/percabangan yang ada pada kode dan nama fungsi OpenCV yang dipakai.

1. **Flowchart A: Kalibrasi** — `load calibration images` → loop semua foto → `findChessboardCorners` → keputusan berhasil? → `cornerSubPix` → kumpulkan titik 3D/2D → jumlah valid cukup? → `calibrateCamera` → tampilkan \(f_x,f_y,c_x,c_y\) dan error → simpan parameter.
2. **Flowchart B: Stereo-to-Depth** — baca pasangan → cek resolusi → `undistort` → `ORB + matching` → keputusan alignment baik? → `StereoSGBM.compute` → disparity valid? → \(Z=f_x B/d\) → estimasi ROI → perbandingan meteran → hasil/error.

**Ketentuan:** gunakan simbol **oval** (Start/End), **jajaran genjang** (Input/Output), **persegi panjang** (Proses), dan **belah ketupat** (Decision). Pada panah dari decision tulis **Ya / Tidak**. Cantumkan juga satuan \(f_x\) (px), \(B\) (m), \(d\) (px), \(Z\) (m).

**Deliverable:** (a) dua flowchart, (b) screenshot sudut chessboard terdeteksi, (c) matriks \(K\) + nilai \(f_x\) hasil kalibrasi + error, (d) foto L/R, (e) disparity/depth map, (f) tabel pengukuran dan analisis error.

**Penilaian yang disarankan (100 poin):** kalibrasi 25; ketelitian pengambilan stereo 20; disparity & depth 20; evaluasi error 15; flowchart 20.

### Contoh flowchart awal (harus dikembangkan)

```mermaid
flowchart TD
    A([Mulai]) --> B[/Baca foto chessboard/]
    B --> C[Deteksi sudut dengan OpenCV]
    C --> D{Sudut ditemukan?}
    D -- Tidak --> E[Ambil atau pilih foto lain]
    E --> B
    D -- Ya --> F[Tambahkan titik 3D dan 2D]
    F --> G{Jumlah foto valid cukup?}
    G -- Tidak --> B
    G -- Ya --> H[calibrateCamera]
    H --> I[/Simpan K, distorsi, dan error/]
    I --> J([Selesai])
```

**Catatan:** flowchart stereo harus dibuat sendiri. Tambahkan mekanisme pengulangan jika validasi alignment gagal.

## 11 — Referensi & batasan ilmiah

- OpenCV Camera Calibration: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html
- OpenCV Depth Map from Stereo Images: https://docs.opencv.org/4.x/dd/d53/tutorial_py_depthmap.html
- OpenCV camera calibration & stereo model: https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html

**Batasan proyek:** ini demonstrasi *two-view stereo* dengan satu HP digeser. Satu kamera dikalibrasi pada posisi yang berbeda waktu; *rig* tidak benar-benar memiliki dua kamera terkalibrasi bersama. Perhitungan depth metrik mengasumsikan kamera tetap menghadap arah sama, pergeseran murni sepanjang sumbu horizontal kamera dan baseline akurat. Jika HP berotasi (atau bergerak maju/mundur), **kalibrasi intrinsik saja tidak cukup**; lakukan estimasi pose relatif + stereo rectification yang benar, atau ulangi pengambilan menggunakan rel yang lebih baik. Untuk peta 3D berkualitas penelitian, gunakan metode estimasi pose dan bundle adjustment/struktur dari gerak dengan validasi tambahan.